In [1]:
file = "AAPL.csv"

stock_symbol = "aapl" #e.g. tsla, msft
column = 'Tweet' # which column to clean

output_filename = "AAPL_cleaned.csv"

# Combine csv files (if needed)

'''ensure that all files csv files you want to combine are in the same folder '''

import os
import glob
import pandas as pd

path = '#path'
extension = 'csv'
os.chdir(path)
output_filename = path.split('/')[-1] + '_combined.csv'

all_files = [i for i in glob.glob('*.{}'.format(extension))]

combined_csv = pd.concat([pd.read_csv(f) for f in all_files])
combined_csv.to_csv(output_filename, index = False, encoding = 'utf-8-sig')

# Cleaning text

In [2]:
import re
import nltk
import numpy as np
import pandas as pd
from nltk import word_tokenize
from nltk.tokenize import RegexpTokenizer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

In [3]:
# text cleaning
def text_lower(text):
    return ' '.join([t.lower() for t in text.split()])

def remove_hashtag_mentions_urls(text):
    return re.sub(r"(?:\@|\#|https?\://)\S+", "", text)

def remove_emoji(text):
    emoji_pattern = re.compile("["
    u"\U0001F600-\U0001F64F"  # emoticons
    u"\U0001F300-\U0001F5FF"  # symbols & pictographs
    u"\U0001F680-\U0001F6FF"  # transport & map symbols
    u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
    u"\U00002702-\U000027B0"
    u"\U000024C2-\U0001F251"
    "]+", flags=re.UNICODE)

    return emoji_pattern.sub(r'', text)

def tokenization(text):
#     word_tokenizer = RegexpTokenizer(r'[-\'\w]+')
    word_tokenizer = RegexpTokenizer(r'\w+')
    tokenized_text = word_tokenizer.tokenize(text)
    return tokenized_text

def remove_numbers(text):
    return [t for t in text if not t.isdigit()]

def remove_stopwords(text, symbol):
    word_dictionary = {
        'aapl': ['$AAPL', 'apple'],
        'googl': ['$GOOGL', 'google'],
        'msft': ['$MSFT', 'microsoft'],
        'tsla': ['$TSLA', 'tesla'],
        'amzn': ['$AMZN', 'amazon'],
        'vxrt': ['$VXRT', 'vaxart'],
        'rui': ['$RUI', 'russell', 'index'],
        'ark': ['$ARK', 'resources'],
        'ater': ['$ATER', 'Aterian']
    }
    stop_words = stopwords.words('english')
    stop_words.append(symbol)
    
    try:
        for word in word_dictionary[symbol]:
            stop_words.append(word)
    except:
        pass 
        
    text_stopremoved = [w for w in text if w not in stop_words]
    return text_stopremoved

def lemmatization(text):
    lemma_word = []
    wordnet_lemmatizer = WordNetLemmatizer()
    for w in text:
        word1 = wordnet_lemmatizer.lemmatize(w, pos = "n")
        word2 = wordnet_lemmatizer.lemmatize(word1, pos = 'v')
        word3 = wordnet_lemmatizer.lemmatize(word2, pos = ('a'))
        lemma_word.append(word3)
    return ' '.join(lemma_word)
    

def clean_text(text, symbol):
    text = text_lower(text)
    text = remove_hashtag_mentions_urls(text)
    text = remove_emoji(text)
    text = tokenization(text)
    text = remove_numbers(text)
    text = remove_stopwords(text, symbol)
    text = lemmatization(text)
    return text 

# Comment Classifier

In [4]:
def comment_classifier(file, column, stock_symbol): 
    df = pd.read_csv(file)
    
    #cleaning text
    text = df[column].to_list()
    cleaned_text = [clean_text(t, stock_symbol) if len(t) > 0 else '' for t in text]
    
    output = 'cleaned ' + column
    df[output] = cleaned_text
    
    #remove missing values from updated df 
    df[output].replace('', np.nan, inplace = True)
    df.dropna(subset=[output], inplace = True)
    
    #for twitter
    try:
        #df = df[df[output].apply(lambda x: len(x)>4)]
        df = df[df['Number of Retweets'] > 5]
        df = df[df['Number of Likes'] > 200]
        df = df[df['Number of Followers'] > 50]
        df = df[df[output].str.split().str.len() > 4]

    except:
        pass
    
    
    return df

In [5]:
df = comment_classifier(file, column, stock_symbol)

df.head(3)

,Unnamed: 0,Username,Tweet,Datetime,Number of Followers,Number of Comments,Number of Retweets,Number of Likes,cleaned Tweet
1,1,TheMoonCarl,Returns over the last 10 Years:\r\n\r\nBitcoin...,2021-09-30 18:01:19+00:00,559219,215,442,2532,return last year bitcoin btc tesla tsla nvidia...
2,2,BrianFeroldi,This thread is just a broad overview of the ba...,2021-09-30 15:30:03+00:00,213150,6,17,215,thread broad overview balance sheet want speci...
3,3,DecadeInvestor,"In 2020, Tim Cook, CEO of $AAPL, earned $265,0...",2021-09-30 13:36:10+00:00,33383,21,68,361,tim cook ceo earn total compensation warren bu...


In [6]:
df.to_csv(output_filename)